In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import * 
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [3]:
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import Adam

In [4]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using {device}')

Using cpu


In [5]:
import requests
from io import BytesIO
import certifi

In [6]:
# 전처리
path = 'https://bit.ly/ds_boston_csv'
response = requests.get(path,verify=certifi.where(),timeout=15)
response.raise_for_status()
data = pd.read_csv(BytesIO(response.content))

# x, y 분할
target = 'medv'
features = ['lstat', 'ptratio', 'crim']
x = data.loc[:, features]
y= data.loc[:, target]

# xtrain, val 분할
x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=.2, random_state=20)

# 스케일러 선언
scaler = MinMaxScaler()
x_train = scaler.fit_transform(x_train)
x_val = scaler.transform(x_val)

In [7]:
# 텐서 변환 + Data Loader 생성 함수
def make_DataSet(x_train, x_val, y_train, y_val, batch_size= 32) :
    x_train_tensor = torch.tensor(x_train, dtype= torch.float32)
    y_train_tensor = torch.tensor(y_train, dtype= torch.float32).view(-1,1)
    x_val_tensor = torch.tensor(x_val, dtype= torch.float32)
    y_val_tensor = torch.tensor(y_val, dtype= torch.float32).view(-1,1)
    
    # TensorDataSet 생성
    train_dataset = TensorDataset(x_train_tensor, y_train_tensor)

    # DataLoader 생성
    train_loader = DataLoader(train_dataset, batch_size= batch_size, shuffle=True)
    return train_loader , x_val_tensor, y_val_tensor

In [8]:
# 넘파이 배열변환 - cpu 연산
y_train = y_train.values
y_val = y_val.values